In [0]:
#display the data from the table company
display(spark.read.table("plworkforce_catalog.001_bronze.company"))

from pyspark.sql import functions as F

# Use silver schema
spark.sql("USE plworkforce_catalog.002_silver")

# Read from Bronze
df = spark.table("plworkforce_catalog.001_bronze.company")

# Transform
df_clean = (df

    # Clean text
    .withColumn("company_name", F.initcap(F.trim("company_name")))
    .withColumn("industry", F.initcap(F.trim("industry")))
    .withColumn("country", F.upper(F.trim("country")))

    # Convert date
    .withColumn("established_date", F.to_date("established_date"))

    # Boolean fix
    .withColumn("is_active", F.col("is_active").cast("boolean"))

    # Data quality
    .dropDuplicates(["company_id"])
    .filter(F.col("company_id").isNotNull())
)

# Write to Silver
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("company")
)

print("Silver company created")

#to display the data in silver company table
display(spark.read.table("plworkforce_catalog.002_silver.company"))